# Waste Detection ADN Pipeline - Training & Inference Notebook

**Hướng dẫn dành cho Contributor / Người chạy thực nghiệm:**
Notebook này chứa toàn bộ các bước tự động để huấn luyện và đánh giá mô hình kiến trúc Hybrid (YOLO + CNN).
Để chạy nghiệm thức với các tham số khác nhau (đổi `learning_rate`, thuật toán augmentation, v.v.), bạn KHÔNG cần sửa code trong Notebook này. Chỉ cần:
1. Chỉnh sửa tham số trong các file YAML thuộc thư mục `configs/` trên Github.
2. Chạy lần lượt các Cell dưới đây.

## Bước 1: Khởi tạo Môi trường (Cài đặt Code & Thư viện)

In [ ]:
!rm -rf Waste-Detection-and-Classification # Xóa thư mục cũ nếu chạy lại nhiều lần
!git clone -b feature/hybrid-finetune https://github.com/nnnhuan251-hcmus/Detect-Waste-ADN.git Waste-Detection-and-Classification

import os
os.chdir('/kaggle/working/Waste-Detection-and-Classification')

# Checkout sang nhánh tích hợp của nhóm (chỉnh sửa lại tên nhánh nếu cần)
!git checkout feature/our-pipeline-integration

!pip install -r requirements.txt
!pip install roboflow
!pip install -e .

## Bước 2: Tải Trực Tiếp 2 Bộ Dữ Liệu & Tiền Xử Lý

In [ ]:
import kagglehub
from roboflow import Roboflow

# 1. Tải bộ dữ liệu chính (TACO) từ Kaggle
print("Đang tải bộ dữ liệu TACO...")
path_taco = kagglehub.dataset_download("wilmacv251/taco-dataset-waste")
!mkdir -p data/raw/TACO
!cp -r {path_taco}/* data/raw/TACO/

# 2. Tải bộ dữ liệu thứ 2 (Fine-tune/Test) từ Roboflow
print("\nĐang tải bộ dữ liệu Roboflow...")
rf = Roboflow(api_key="5r1QbIwiFST8QcGoDpot")
project = rf.workspace("rf100-vl").project("taco-trash-annotations-in-context-dtyly-awiq")
version = project.version(2)
dataset = version.download("coco")

# (Tùy chọn) Chuyển dữ liệu Roboflow vào thư mục raw chuẩn của pipeline
!mkdir -p data/raw/Roboflow_Dataset
!cp -r {dataset.location}/* data/raw/Roboflow_Dataset/

print("\n--- BẮT ĐẦU CHẠY PIPELINE TIỀN XỬ LÝ CHO DATASET CHÍNH ---")
# Chạy pipeline tạo bộ dữ liệu 7 nhãn và phân chia Train/Val/Test cho TACO
!PYTHONPATH=src python scripts/data/prepare_taco.py --data-config configs/data/taco_7class.yaml

## Bước 3: Huấn luyện Mô hình Giai đoạn 1 (YOLOv8 Detector)

In [ ]:
!PYTHONPATH=src python scripts/train/train_detector.py \
  --data-config configs/data/taco_7class.yaml \
  --model-config configs/models/hybrid_yolov8n_effb0.yaml \
  --experiment-config configs/experiments/run1_baseline.yaml

## Bước 4: Huấn luyện Mô hình Giai đoạn 2 (EfficientNet-B0 Classifier)

In [ ]:
!PYTHONPATH=src python scripts/train/train_classifier.py \
  --data-config configs/data/taco_7class.yaml \
  --model-config configs/models/hybrid_yolov8n_effb0.yaml \
  --experiment-config configs/experiments/run1_baseline.yaml

## Bước 5: Đánh giá Toàn trình (End-to-End Evaluation)

In [ ]:
!PYTHONPATH=src python scripts/eval/evaluate_hybrid.py \
  --data-config configs/data/taco_7class.yaml \
  --model-config configs/models/hybrid_yolov8n_effb0.yaml \
  --detector-weights outputs/checkpoints/hybrid_yolov8n_effb0/run1_baseline/weights/best.pt \
  --classifier-weights outputs/checkpoints/efficientnet_b0/run1_baseline/best.pth \
  --split test

## Bước 6: Phân tích Lỗi và Vẽ Biểu đồ Đối chứng (XAI & Ablation)

In [ ]:
# Vẽ bản đồ nhiệt Grad-CAM trên 1 ảnh mặc định
!PYTHONPATH=src python scripts/eval/run_xai.py \
  --detector outputs/checkpoints/hybrid_yolov8n_effb0/run1_baseline/weights/best.pt \
  --classifier_weights outputs/checkpoints/efficientnet_b0/run1_baseline/best.pth

# Vẽ biểu đồ so sánh Ablation
!PYTHONPATH=src python scripts/eval/ablation_comparison.py \
  --image data/processed/yolo_binary_waste/images/test/000000.jpg \
  --detector outputs/checkpoints/hybrid_yolov8n_effb0/run1_baseline/weights/best.pt \
  --classifier_weights outputs/checkpoints/efficientnet_b0/run1_baseline/best.pth

In [ ]:
# Xem trực tiếp ảnh kết quả trên Notebook
from IPython.display import Image, display

try:
    print("--- 1. BẢN ĐỒ NHIỆT GRAD-CAM (XAI) ---")
    display(Image(filename='test_heatmap_gradcam.png'))
except Exception as e:
    print("Chưa có ảnh Grad-CAM.")

try:
    print("\n--- 2. KẾT QUẢ SO SÁNH ABLATION STUDY ---")
    display(Image(filename='ablation_comparison_result.png'))
except Exception as e:
    print("Chưa có ảnh Ablation.")